# This is a simple example script for 2D x/y stitching using muvis-align and the multiview-stitcher package

In [ ]:
import sys
sys.path.append('..')

from multiview_stitcher import spatial_image_utils as si_utils, param_utils
from multiview_stitcher import vis_utils
import networkx as nx
import numpy as np

from src.muvis_align.MVSRegistration import MVSRegistration
from src.muvis_align.image.util import get_sim_physical_size, show_image

## Initialise muvis-align, initialise sims, and pre-process

In [ ]:
reg = MVSRegistration(operation='register', input_path='../data/S000/*.zarr', output_path='../../output/', ui='mpl')
sims = reg.init_sims()
reg_sims, reg_indices, _ = reg.preprocess(sims)

for label, sim in zip(reg.file_labels, sims):
    print(label, si_utils.get_origin_from_sim(sim), get_sim_physical_size(sim))

## Initialise registration parameters

In [ ]:
register_params = {
	'pairing': 'orthogonal',
	'transform_type': 'rigid',
	'method': 'sift',
	'gaussian_sigma': 2,
	'normalisation': True,
	'max_keypoints': 5000,
	'inlier_threshold_factor': 0.05,
	'max_trials': 1000,
	'ransac_iterations': 3,
    'metrics': ['ncc', 'ssim', 'onmi'],
	'n_parallel_pairwise_regs': 1,
}

## Perform pair registration (using multiview-stitcher)

In [ ]:
reg_results = reg.register_pairs(sims, register_sims=reg_sims, params=register_params)

## Show pair metrics

In [ ]:
reg_results['metrics']

## Pair transforms modification

In [ ]:
pairs_graph = reg_results['pairs_graph']
transforms = nx.get_edge_attributes(pairs_graph, 'transform')
qualities = nx.get_edge_attributes(pairs_graph, 'quality')
transforms[0, 1] = param_utils.identity_transform(2)    # modify first transform to eye transform
qualities[0, 1] = np.array(1)    # set quality to 1
nx.set_edge_attributes(pairs_graph, transforms, 'transform')
nx.set_edge_attributes(pairs_graph, qualities, 'quality')

## Perfrom global registration (using multiview-stitcher)

In [ ]:
%matplotlib inline
msims = reg_results['msims']
reg_results2 = reg.register_global(sims, msims, register_indices=reg_indices, params=register_params)

## Show global metrics

In [ ]:
reg_results2['metrics']

## Show registration mapping

In [ ]:
mappings = reg_results2['mappings']
for key, mapping in mappings.items():
    print(f'{reg.file_labels[key]}:\n', mapping.sel(t=0).data)

## Visualise registered sims

In [ ]:
%matplotlib inline
fig, ax = vis_utils.plot_positions(sims, transform_key=reg.reg_transform_key, use_positional_colors=False, view_labels=reg.file_labels)

## Perform fusion (using multiview-stitcher)

In [ ]:
%matplotlib inline
fused, _ = reg.fuse(sims)

fused

## Output fused result

In [ ]:
%matplotlib inline
show_image(fused[0, 0])

In [ ]:
reg.save('stitching2d', fused)